# Example 7.11 / 1: Formulate a Knapsack

Companion notebook for **7.11 Knapsack, Covering and Packing** in the AMPL lab Documentation, section "Formulate a Knapsack". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Formulate a Knapsack

Pick items of given value and weight so that the total weight fits the capacity:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS;                        # candidate items
param value {ITEMS};              # value gained if the item is taken
param weight {ITEMS} >= 0;        # weight of each item
param capacity >= 0;              # knapsack weight limit
var Take {ITEMS} binary;          # 1 if the item is packed

maximize Value: sum {i in ITEMS} value[i] * Take[i];  # total value packed
# packing row: weight of chosen items fits the capacity
subject to Knapsack:
    sum {i in ITEMS} weight[i] * Take[i] <= capacity;
# two valid inequalities, dropped or kept below
subject to MinimalCover: Take["b"] + Take["c"] + Take["d"] <= 2;
subject to ExtendedCover: sum {i in ITEMS} Take[i] <= 2;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ITEMS"] = ["a", "b", "c", "d"]
ampl.param["value"] = {"a": 12, "b": 11, "c": 9, "d": 4}
ampl.param["weight"] = {"a": 6, "b": 5, "c": 4, "d": 2}
ampl.param["capacity"] = 10

In [5]:
# run.py
ampl.con["MinimalCover"].drop()
ampl.con["ExtendedCover"].drop()
ampl.solve(solver="highs")
print(ampl.var["Take"].to_dict())

HiGHS 1.14.0: optimal solution; objective 21
0 simplex iterations
0 branching nodes
{'a': 1, 'b': 0, 'c': 1, 'd': 0}


### Pure AMPL: data.dat and model.run

Items `a` and `c` fill the capacity exactly and are worth 21.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param: ITEMS: value weight := a 12 6  b 11 5  c 9 4  d 4 2;
param capacity := 10;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
drop MinimalCover;
drop ExtendedCover;
solve;
display Take;

HiGHS 1.14.0: optimal solution; objective 21
0 simplex iterations
0 branching nodes
Take [*] :=
a  1
b  0
c  1
d  0
;



### Another way in amplpy

Fractional selections make the relaxation optimistic. A **cover** is a set of items whose total
weight exceeds the capacity, so at least one of them must be left out: `b`, `c` and `d` weigh 11,
which gives `MinimalCover`. Every three items here weigh at least 11, so the stronger
`ExtendedCover`, at most two items in all, is also valid. With the same `model.mod` and `data.py`,
compare the relaxation bound with no cut, each cut, and the integer optimum:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["ITEMS"] = ["a", "b", "c", "d"]
ampl.param["value"] = {"a": 12, "b": 11, "c": 9, "d": 4}
ampl.param["weight"] = {"a": 6, "b": 5, "c": 4, "d": 2}
ampl.param["capacity"] = 10

In [11]:
# run.py
ampl.con["MinimalCover"].drop()
ampl.con["ExtendedCover"].drop()
ampl.option["relax_integrality"] = 1
ampl.solve(solver="highs", verbose=False)
print("no cut          ", ampl.obj["Value"].value())
for cut in ["MinimalCover", "ExtendedCover"]:
    ampl.con[cut].restore()
    ampl.solve(solver="highs", verbose=False)
    print(f"{cut:16}", ampl.obj["Value"].value())
    ampl.con[cut].drop()
ampl.option["relax_integrality"] = 0
ampl.solve(solver="highs", verbose=False)
print("integer optimum ", ampl.obj["Value"].value())

no cut           22.0
MinimalCover     22.0
ExtendedCover    21.5
integer optimum  21.0
